# Production RAG Deployment

This notebook covers deploying RAG systems to production.

## FastAPI Server

In [ ]:
# main.py - FastAPI application

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List

app = FastAPI(title="RAG API")

class QueryRequest(BaseModel):
    question: str
    use_cache: bool = True
    top_k: Optional[int] = 4

class QueryResponse(BaseModel):
    answer: str
    sources: List[dict]
    cached: bool = False

# Initialize RAG
from rag_pipeline import create_rag_pipeline
rag = create_rag_pipeline()

@app.post("/query", response_model=QueryResponse)
async def query(request: QueryRequest):
    """Query the RAG system."""
    try:
        result = rag.query(
            question=request.question,
            use_cache=request.use_cache,
            top_k=request.top_k
        )
        return QueryResponse(**result)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health():
    """Health check."""
    return {"status": "healthy"}

## Docker Configuration

In [ ]:
# Dockerfile

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
# docker-compose.yml

version: '3.8'

services:
  rag-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
      - REDIS_URL=redis://redis:6379
    depends_on:
      - redis
    deploy:
      resources:
        limits:
          memory: 2G
          cpus: '1.0'

  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
    volumes:
      - redis-data:/data

volumes:
  redis-data:

## Environment Configuration

In [ ]:
# .env.example

# API Keys
OPENAI_API_KEY=sk-...

# Vector Database
PINECONE_API_KEY=...
PINECONE_ENV=us-west-2

# Redis Cache
REDIS_URL=redis://localhost:6379

# Application
LOG_LEVEL=INFO
CACHE_TTL=3600

## Testing

In [ ]:
# test_rag.py

import pytest
from rag_pipeline import create_rag_pipeline

@pytest.fixture
def rag():
    return create_rag_pipeline()

def test_rag_query(rag):
    """Test basic query."""
    result = rag.query("What is RAG?")
    
    assert "answer" in result
    assert "sources" in result
    assert len(result["answer"]) > 0

def test_empty_query(rag):
    """Test empty query handling."""
    with pytest.raises(ValueError):
        rag.query("")

def test_caching(rag):
    """Test caching works."""
    query = "What is RAG?"
    
    result1 = rag.query(query, use_cache=True)
    result2 = rag.query(query, use_cache=True)
    
    assert result1["answer"] == result2["answer"]
    assert result2["cached"] == True

## Monitoring

In [ ]:
# monitoring.py

from prometheus_client import Counter, Histogram, generate_latest
from fastapi import Response

# Metrics
REQUEST_COUNT = Counter('rag_requests_total', 'Total RAG requests')
REQUEST_LATENCY = Histogram('rag_request_latency_seconds', 'Request latency')
CACHE_HITS = Counter('rag_cache_hits_total', 'Cache hits')
ERROR_COUNT = Counter('rag_errors_total', 'Total errors')

@app.middleware("http")
async def track_metrics(request, call_next):
    """Track request metrics."""
    import time
    
    start = time.time()
    try:
        response = await call_next(request)
        REQUEST_COUNT.inc()
        return response
    except Exception:
        ERROR_COUNT.inc()
        raise
    finally:
        REQUEST_LATENCY.observe(time.time() - start)

@app.get("/metrics")
async def metrics():
    """Expose Prometheus metrics."""
    return Response(generate_latest(), media_type="text/plain")

## Next Steps

1. Add authentication
2. Set up CI/CD pipeline
3. Configure auto-scaling
4. Add detailed logging
5. Set up alerts